# 06 — Composition gene by gene

Each genome is aligned to its subtype's annotated reference and cut into the 10 RSV genes, so we can see which gene carries the signal. Alignment keeps insertions, so the ON1/BA G-gene duplications are retained.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)
from rsv import io, composition
from rsv import trends

In [2]:
g = pd.read_csv("../data/processed/rsv_gene_features.tsv", sep="\t")
g = g[(g.year>=2015)&(g.year<=2026)]
print(len(g), "gene-rows;", g["Accession ID"].nunique(), "sequences")
GENES = ["NS1","NS2","N","P","M","SH","G","F","M2","L"]
g.groupby(["Subtype","gene"]).agg(n=("gc_content","size"),
   gc=("gc_content","mean"), oe_CG=("oe_CG","mean")).round(4)

320590 gene-rows; 32059 sequences


n      gc   oe_CG
Subtype gene                       
A       F     15052  0.3525  0.2187
        G     15052  0.4301  0.3263
        L     15052  0.3080  0.2680
        M     15052  0.3661  0.2035
        M2    15052  0.3637  0.4059
        N     15052  0.3697  0.1757
        NS1   15052  0.3047  0.0164
        NS2   15052  0.3422  0.0710
        P     15052  0.3569  0.1555
        SH    15052  0.3188  0.4022
B       F     17007  0.3458  0.1887
        G     17007  0.4046  0.2640
        L     17007  0.3121  0.2413
        M     17007  0.3721  0.3348
        M2    17007  0.3636  0.3136
        N     17007  0.3706  0.1530
        NS1   17007  0.3028  0.2045
        NS2   17007  0.3487  0.5195
        P     17007  0.3756  0.3080
        SH    17007  0.2981  0.2309

### Which genes carry the within-lineage CpG O/E trend?

In [3]:
rows=[]
for (sub,gene),s in g.groupby(["Subtype","gene"]):
    if len(s)<500 or s.Lineage.nunique()<3: continue
    r=trends.decompose_trend(s,"oe_CG"); r.insert(0,"gene",gene); r.insert(0,"Subtype",sub)
    rows.append(r)
gd=pd.concat(rows,ignore_index=True)
from statsmodels.stats.multitest import multipletests
gd["p_within_adj"]=multipletests(gd["p_within"],method="fdr_bh")[1]
gd["gene"]=pd.Categorical(gd["gene"],GENES,ordered=True)
gd.sort_values(["Subtype","gene"])[["Subtype","gene","n","slope_total","slope_within","p_within_adj"]].round(6)

,Subtype,gene,n,slope_total,slope_within,p_within_adj
6,A,NS1,15052,0.001075,0.002658,0.000000
7,A,NS2,15052,0.001067,0.001410,0.000000
5,A,N,15052,0.000953,0.000307,0.000040
8,A,P,15052,-0.002855,0.000757,0.000000
3,A,M,15052,0.000617,0.001224,0.000000
9,A,SH,15052,-0.006641,-0.004581,0.000000
1,A,G,15052,0.000299,-0.001177,0.000000
0,A,F,15052,0.000855,-0.000074,0.346325
4,A,M2,15052,-0.002483,-0.000656,0.000000
2,A,L,15052,-0.000159,0.000107,0.012141
